In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from scipy.spatial.distance import pdist
from evaluation_utils_module import  rename, select_minimal_examples_old
from constants import LAMBDA_CONTS, DIFF_CONTS
def load_gen_direct_data(en, cont, alg, name):
    path = f"./Data/generation_ex/{en}/{cont}/{alg}/{name}"
    json_files = list(Path(path).glob("*.json"))
    print(f"Found {len(json_files)} files!")
    with open(json_files[0], "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

def load_final_direct_data(en, cont, alg, name):
    path = f"./Data/final/{en}/{cont}/{alg}/{name}"
    json_files = list(Path(path).glob("*.json"))
    print(f"Found {len(json_files)} files!")
    with open(json_files[0], "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

def load_grid_direct_data(en, cont, alg, name):
    path = f"./Data/grid_search/{name}/{en}/{cont}/{alg}"
    json_files = list(Path(path).glob("*.json"))
    print(f"Found {len(json_files)} files!")
    with open(json_files[0], "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

def gen_extract_values(file_content):
    df = pd.DataFrame(file_content["fitness"]).reset_index(names="seed").melt(
        id_vars="seed", 
        var_name="ng", 
        value_name="population"
    )
    df["ng"] = df["ng"].astype(int)
    df["max_fit"] = df["population"].apply(lambda x: np.max(list(map(lambda y: y[0], x))))
    df["avg_fit"] = df["population"].apply(lambda x: np.mean(list(map(lambda y: y[0], x))))
    df["median_fit"] = df["population"].apply(lambda x: np.median(list(map(lambda y: y[0], x))))
    df["min_fit"] = df["population"].apply(lambda x: np.min(list(map(lambda y: y[0], x))))
    df["std_fit"] = df["population"].apply(lambda x: np.std(list(map(lambda y: y[0], x))))

    df["div"] = df["population"].apply(lambda x: np.mean(pdist(list(map(lambda y: y[1], x)))))
    return df



def final_extract_values(file_content):
    df = pd.DataFrame(file_content["fitness"]).reset_index(names="seed").melt(
        var_name="seed", 
        value_name="population"
    )
    df["max_fit"] = df["population"].apply(lambda x: np.max(list(map(lambda y: y[0], x))))
    df["avg_fit"] = df["population"].apply(lambda x: np.mean(list(map(lambda y: y[0], x))))
    df["median_fit"] = df["population"].apply(lambda x: np.median(list(map(lambda y: y[0], x))))
    df["min_fit"] = df["population"].apply(lambda x: np.min(list(map(lambda y: y[0], x))))
    df["std_fit"] = df["population"].apply(lambda x: np.std(list(map(lambda y: y[0], x))))
    df["div"] = df["population"].apply(lambda x: np.mean(pdist(list(map(lambda y: y[1], x)))))

    return df

def get_protocol_table2(cont, protocol):
    origin, index = select_minimal_examples_old(protocol["origin"])
    df_template = {
        cont:{
            "origin":rename(origin[0]),
            "final": protocol["final"]
        }
    }
    if cont == "novelty_limit":
        print(protocol["origin"])
        print(protocol["final"])
    df = pd.concat(
        {
            row: pd.Series({
                (lvl1, lvl2): value
                for lvl1, sub in vals.items()
                for lvl2, value in sub.items()
            })
            for row, vals in df_template.items()
        },
        axis=1
    ).T

    df.columns = pd.MultiIndex.from_tuples(df.columns)
    return df

def get_protocol_table(en, alg, name):
    big_table = pd.DataFrame()
    cts = LAMBDA_CONTS if alg == "lambda" else DIFF_CONTS
    for cont in cts:
        protocol = load_grid_direct_data(en, cont, alg, name)
        table = get_protocol_table2(cont, protocol)
        big_table = pd.concat([big_table, table])

    return big_table.sort_index(axis=1)   


2026-06-28 18:39:14.948676: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-28 18:39:14.985590: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-28 18:39:16.016389: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .auto

# Grid protocols

In [12]:
ens = ["lunarlander", "cartpoles"]
en = "cartpole"
alg = "lambda"
df = get_protocol_table(en, alg, "server_try0")
df.to_csv(f"./Data/grid_protocol/{en}_{alg}.csv")

Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
Found 1 files!
[{'crossmethod': 'uniform', 'lambda': 20, 'mu': 30, 'mutation_rate': 0.06, 'cross_rate': 1.0, 'sigma': 1.0, 'archiving_period': 4, 'archive_batch': 5, 'limit': 220.0, 'cross': 0.6}]
{'cross_method': 'uniform', 'l': 25, 'm': 30, 'mr': 0.009999999999999995, 'cr': 1.0, 'mutation_sigma': 1.0, 'archiving_period': 4, 'archive_batch': 5, 'limit': 220.0, 'cross_uni': 0.6}


In [15]:
from tabulate import tabulate
df2 = pd.read_csv(f"./Data/grid_protocol/{en}_{alg}.csv", header=[0,1], index_col=[0])
if alg =="lambda":
    df2=df2.drop([("final","cross_method"), ("origin", "cross_method")], axis=1)
delta =  df2["final"] - df2["origin"]
delta = delta.loc[~(delta.fillna(0) == 0).all(axis=1)]
#delta = delta.loc[~(delta.fillna(0) == 0).all()]
mask = ~delta.fillna(0).eq(0).all(axis=0)
delta = delta.loc[:, mask]

delta.to_csv(f"./Data/grid_protocol/delta_{en}_{alg}.csv")
latex = delta.to_latex(
    index=True,
    escape=True,
    float_format="%.3f",
    column_format="l" + "c" * len(df.columns)
)
print(latex)


\begin{tabular}{lcccccccccccccccccccccccc}
\toprule
{} &     cr &   l &   m &     mr \\
\midrule
fitness           & -0.100 &   5 &   5 &  0.000 \\
novelty           &  0.000 &   5 &  -5 & -0.050 \\
sub\_novelty       &  0.000 &  10 & -10 &  0.000 \\
fit\_archiving     & -0.100 &  -5 &  10 &  0.000 \\
elite\_archiving   & -0.100 &   0 &   0 &  0.000 \\
novelty\_archiving & -0.100 &   5 &   5 & -0.050 \\
novelty\_limit     &  0.000 &   5 &   0 & -0.050 \\
\bottomrule
\end{tabular}



/tmp/ipykernel_16505/716156018.py:12: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  latex = delta.to_latex(


In [4]:
df = gen_extract_values(load_gen_direct_data("cartpole", "fitness", "lambda", "server_try0"))
df.groupby("ng").median()

Found 1 files!


/tmp/ipykernel_16505/2083778437.py:2: FutureWarning: The default value of numeric_only in DataFrameGroupBy.median is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  df.groupby("ng").median()


,max_fit,avg_fit,median_fit,min_fit,std_fit,div
ng,,,,,,
5,380.0,248.844444,182.166667,38.333333,137.925233,0.547401
10,500.0,320.355556,380.000000,23.666667,150.994717,0.391541
15,500.0,238.411111,181.500000,14.166667,152.964761,0.386958
20,500.0,215.944444,245.666667,9.833333,136.345529,0.549736
25,500.0,223.366667,224.000000,25.500000,144.079880,0.527765


In [5]:
file_content = load_final_direct_data("cartpole", "fitness", "lambda", "server_try0")
df = pd.DataFrame(file_content["fitness"]).reset_index(names="seed").melt(
        var_name="seed", 
        value_name="population"
    )
df

Found 1 files!


,seed,population
0,seed,0
1,seed,1
2,seed,2
3,seed,3
4,seed,4
...,...,...
685,108,"[[500.0], [-0.03188566925625007, 0.00953948143..."
686,108,"[[248.5], [0.43245350321133935, 0.503346965643..."
687,108,"[[248.5], [0.43245350321133935, 0.503346965643..."
688,108,"[[96.16666666666667], [0.09165313715736072, 0...."
